# Interview Prep: 17 - API Design & Architecture

Modern engineering is built on APIs. This notebook covers how to choose between protocols and how to design APIs that are resilient, versioned, and idempotent.

---

## 1. Protocol Comparison: REST vs. GraphQL vs. gRPC

### **Interview Question**: "When would you use GraphQL instead of REST, and what are the trade-offs?"

**Answer**:
- **GraphQL**: Best for complex UIs where the client needs specific data shapes (avoids over-fetching). It allows fetching multiple resources in one request.
- **REST**: Best for public APIs, simple CRUD, and when caching at the CDN/HTTP level is a priority.

| Feature | REST | GraphQL | gRPC |
| :--- | :--- | :--- | :--- |
| **Data Format** | JSON (Text) | JSON (Text) | Protobuf (Binary) |
| **Contract** | Loose (OpenAPI) | Strict (Schema) | Strict (Proto File) |
| **Caching** | Excellent (HTTP) | Difficult (Post Based) | Difficult (Custom) |
| **Best Use Case** | External/Mobile APIs | Complex Data Dashboards | Internal Microservices |

---

## 2. API Versioning Strategies

### **Interview Question**: "How do you handle breaking changes in a public API?"

**Answer**: Versioning. 
1. **URL Versioning** (e.g., `/api/v1/users`): Most common. Visual and easy to route.
2. **Header Versioning** (e.g., `X-API-Version: 1`): Keeps URLs clean, but harder to cache and test via browser.

**Senior Tip**: Always use a deprecation schedule. Warn users via headers (e.g., `Sunset` or `Deprecated`) months before shutting down a version.

---

## 3. Idempotency Mastery

### **Interview Question**: "Design a payment API that prevents double-charging if the user clicks 'Pay' twice."

**The Solution**: **Idempotency Keys**.
- The client generates a unique `X-Idempotency-Key` (UUID) and sends it in the header.
- On the server, we check if we've seen this key before (usually in Redis).
- If yes, return the cached result of the first call.
- If no, process the payment and store the result under that key.

---

## 4. Coding Challenge: Idempotent API with FastAPI

**Task**: Implement a simplified version of an idempotency check in a FastAPI endpoint.


In [ ]:
from fastapi import FastAPI, Header, HTTPException
from typing import Optional
import uuid

app = FastAPI()

# Mock simple Redis-style store
idempotency_store = {}

@app.post("/process-payment")
async def process_payment(amount: float, x_idempotency_key: str = Header(...)):
    # 1. Check if key exists
    if x_idempotency_key in idempotency_store:
        return {"status": "repeated", "result": idempotency_store[x_idempotency_key], "msg": "Duplicate request blocked"}
    
    # 2. Simulate heavy processing
    charge_id = f"ch_{uuid.uuid4().hex[:8]}"
    result = {"charge_id": charge_id, "amount": amount}
    
    # 3. Store result
    idempotency_store[x_idempotency_key] = result
    
    return {"status": "success", "result": result}

print("Idempotency logic for payment API ready.")

## 5. API Security Checklist
- **Rate Limiting**: Prevent DDoS and abuse.
- **Scope-based Auth**: Ensure a client can only access what they need.
- **Input Validation**: Use schemas (Pydantic/Marshmallow) to block malicious payloads.
---